# HPO Visualization: Model Performance Analysis
This notebook provides a production-grade interface to visualize forecasting model performance from the Hyperparameter Optimization (HPO) stage. 
It automatically handles path resolution, model instantiation, and inference to compare predicted values against ground truth across different asset classes.

### How to use:
Simply call the main function at the bottom of the notebook:
`plot_pred(model_name="DLinear", horizon=24, trial=0, plot_samples=100)`

In [ ]:
import sys
import json
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import (
    mean_squared_error, 
    mean_absolute_error, 
    r2_score,
    mean_absolute_percentage_error
)

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models import get_model_class
from src.utils.config import ProjectConfig
from src.data.windowing import load_processed_data
from src.training.trainer import Trainer
from src.utils.seed import get_device

# Set plotting style
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

### 📌 Configuration & Utilities
The following section initializes the project configuration and defines internal helper functions for path resolution and data loading.

In [ ]:
# Initialize Project Configuration
try:
    config = ProjectConfig(project_root)
except Exception as e:
    print(f"Error initializing ProjectConfig: {e}")

def _get_hpo_base_path() -> Path:
    """Return the base path for HPO models."""
    return config.get_path("models_dir") / "hpo"

def _get_available_models() -> List[str]:
    """Dynamically discover models that have HPO results."""
    hpo_path = _get_hpo_base_path()
    if not hpo_path.exists():
        return []
    return [d.name for d in hpo_path.iterdir() if d.is_dir()]

def _get_available_horizons(model_name: str) -> List[int]:
    """Discover available horizons for a given model."""
    hpo_path = _get_hpo_base_path() / model_name
    horizons = set()
    for cat_dir in hpo_path.iterdir():
        if cat_dir.is_dir():
            for asset_dir in cat_dir.iterdir():
                if asset_dir.is_dir():
                    for horizon_dir in asset_dir.iterdir():
                        if horizon_dir.is_dir() and horizon_dir.name.isdigit():
                            horizons.add(int(horizon_dir.name))
    return sorted(list(horizons))

def _get_available_trials(model_name: str, category: str, asset: str, horizon: int) -> List[int]:
    """Discover available trial numbers for a specific configuration."""
    path = _get_hpo_base_path() / model_name / category / asset / str(horizon)
    if not path.exists():
        return []
    trials = []
    for d in path.iterdir():
        if d.is_dir() and d.name.startswith("trial_"):
            try:
                trials.append(int(d.name.split("_")[1]))
            except (IndexError, ValueError):
                continue
    return sorted(trials)

def _load_trial_metadata(model_name: str, category: str, asset: str, horizon: int, trial_num: int) -> Dict[str, Any]:
    """Load hyperparameters and metadata for a specific trial."""
    hpo_dir = _get_hpo_base_path() / model_name / category / asset / str(horizon)
    trials_csv = hpo_dir / "trials.csv"
    
    if not trials_csv.exists():
        raise FileNotFoundError(f"Trials record not found: {trials_csv}")
    
    df = pd.read_csv(trials_csv)
    trial_row = df[df["number"] == trial_num]
    
    if trial_row.empty:
        raise ValueError(f"Trial {trial_num} not found in {trials_csv}")
    
    return trial_row.iloc[0].to_dict()

def _load_best_trial_num(model_name: str, category: str, asset: str, horizon: int) -> int:
    """Load the best trial number from best_trial.json."""
    path = _get_hpo_base_path() / model_name / category / asset / str(horizon) / "best_trial.json"
    if not path.exists():
        raise FileNotFoundError(f"best_trial.json not found at {path}")
    with open(path, "r") as f:
        data = json.load(f)
    return data["trial_number"]

def _load_model_for_trial(model_name: str, category: str, asset: str, horizon: int, trial_num: int):
    """Instantiate and load the best model from a specific HPO trial."""
    hpo_dir = _get_hpo_base_path() / model_name / category / asset / str(horizon)
    trial_dir = hpo_dir / f"trial_{trial_num:03d}"
    checkpoint_path = trial_dir / "model_best.pt"
    
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    
    # 1. Load raw trial metadata
    metadata = _load_trial_metadata(model_name, category, asset, horizon, trial_num)
    
    # 2. Filter out non-model hyperparameters
    excluded_keys = {"number", "value", "state", "datetime_start", "datetime_complete", "duration"}
    raw_hparams = {k: v for k, v in metadata.items() if k not in excluded_keys and not pd.isna(v)}
    
    # 3. Resolve categorical indices if any
    ss_config = config.get_model_search_space(model_name)["search_space"]
    clean_params = {}
    for k, v in raw_hparams.items():
        # Only remap if it ends in _idx AND the base key exists in search space
        # AND the indexed key itself is not in search space (to avoid collisions like target_idx)
        if k.endswith("_idx"):
            original_key = k[:-4]
            if original_key in ss_config and k not in ss_config:
                choices = ss_config[original_key]["choices"]
                clean_params[original_key] = choices[int(v)]
            else:
                clean_params[k] = v
        else:
            clean_params[k] = v
    
    # 4. Separate model vs training params
    model_hparams = {k: v for k, v in clean_params.items() if k not in ("learning_rate", "batch_size")}
    
    # 5. Resolve architecture class
    model_cls = get_model_class(model_name)
    
    # 6. Initialize model
    window_size = config.get_window_size(horizon)
    model = model_cls(
        input_size=len(config.dataset["features"]),
        window_size=window_size,
        horizon=horizon,
        **model_hparams
    )
    
    # 7. Load weights
    device = torch.device("cpu")
    model.load_checkpoint(checkpoint_path, device=device)
    model.eval()
    
    return model, clean_params, device

def _calculate_smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Calculate Symmetric Mean Absolute Percentage Error (SMAPE)."""
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0 # Handle division by zero
    return float(np.mean(diff) * 100)


In [ ]:
def plot_pred(model_name: str, horizon: int, trial: int, plot_samples: int = 150):
    """
    Visualize each forecast horizon step in a separate plot.

    Args:
        model_name: Model architecture name (e.g., 'DLinear', 'LSTM')
        horizon: Forecast horizon (e.g., 4, 24)
        trial: HPO trial number (0-indexed)
        plot_samples: Number of last test samples to visualize
    """

    # 1. Validate Model name
    available_models = _get_available_models()
    if not available_models:
        print("❌ Error: No HPO results found in 'models/hpo/'. Run HPO stage first.")
        return
        
    if model_name not in available_models:
        print(f"❌ Error: Model '{model_name}' not found.")
        print(f"Available models with HPO results: {available_models}")
        return

    # 2. Validate Horizon
    available_horizons = _get_available_horizons(model_name)
    if horizon not in available_horizons:
        print(f"❌ Error: Horizon {horizon} not available for {model_name}.")
        print(f"Available horizons: {available_horizons}")
        return

    # 3. Find categories
    hpo_base = _get_hpo_base_path() / model_name
    categories = []

    for cat_dir in hpo_base.iterdir():
        if cat_dir.is_dir():
            try:
                asset = config.get_representative_asset(cat_dir.name)
                if (cat_dir / asset / str(horizon)).exists():
                    categories.append(cat_dir.name)
            except KeyError:
                continue

    if not categories:
        print(f"❌ Error: No categories found for {model_name} with horizon {horizon}")
        return

    print(f"🔍 Analyzing {model_name} | Horizon={horizon} | Trial={trial}")

    for category in categories:
        representative = config.get_representative_asset(category)

        # 4. Validate Trial
        available_trials = _get_available_trials(
            model_name, category, representative, horizon
        )
        if trial not in available_trials:
            print(f"\n⚠️ Trial {trial} not found for {category.upper()} ({representative}).")
            print(f"   Available trials: {available_trials}")
            continue

        try:
            # 5. Load model and data
            model, params, device = _load_model_for_trial(
                model_name, category, representative, horizon, trial
            )

            data_dir = config.get_path("data_processed") / category / representative / str(horizon)
            if not data_dir.exists():
                print(f"❌ Processed data not found at {data_dir}")
                continue

            dataset = load_processed_data(data_dir)
            test_x, test_y = dataset["test_x"], dataset["test_y"]

            # 6. Generate predictions
            trainer = Trainer(model, device, {"batch_size": 128}, seed=42)
            preds = trainer.predict(test_x)

            max_samples = min(plot_samples, len(test_y))

            y_true_slice = test_y[-max_samples:]
            y_pred_slice = preds[-max_samples:]

            # =========================================
            # Plot EACH horizon step separately
            # =========================================
            for h in range(horizon):

                plt.figure(figsize=(16, 5))

                plt.plot(
                    y_true_slice[:, h],
                    label=f"Ground Truth (t+{h+1})",
                    linewidth=2,
                    alpha=0.9
                )

                plt.plot(
                    y_pred_slice[:, h],
                    linestyle='--',
                    label=f"Prediction (t+{h+1})",
                    linewidth=2,
                    alpha=0.9
                )

                plt.fill_between(
                    range(max_samples),
                    y_true_slice[:, h],
                    y_pred_slice[:, h],
                    alpha=0.2,
                    label="Error"
                )

                plt.title(
                    f"{model_name} | {category.upper()} | {representative}\n"
                    f"Horizon Step: t+{h+1} | Total Horizon={horizon}",
                    fontweight='bold'
                )

                plt.xlabel("Test Time Index (Last Samples)")
                plt.ylabel("Scaled Value")
                plt.legend()
                plt.grid(True, linestyle='--', alpha=0.4)
                plt.tight_layout()
                plt.show()

            # =========================================
            # Plot MAE per Forecast Step
            # =========================================
            step_mae = np.mean(np.abs(test_y - preds), axis=0)

            plt.figure(figsize=(8, 4))
            plt.plot(range(1, horizon + 1), step_mae, marker='o')
            plt.title(
                f"MAE per Forecast Step\n{model_name} | Horizon={horizon}",
                fontweight='bold'
            )
            plt.xlabel("Forecast Step")
            plt.ylabel("MAE")
            plt.grid(True, linestyle='--', alpha=0.5)
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"❌ Error processing {category}/{representative}: {e}")


### 📌 Multi-Model Comparison
The `compare_hpo_models` function allows for a rigorous statistical and metric-based comparison between different architectures selected during the HPO phase. It automatically identifies the best trial for each model and computes comprehensive regression metrics.

In [ ]:
def compare_hpo_models(
    models: List[str],
    horizon: int,
    asset_class: str,
    shift: int = 1,
    plot: bool = True
) -> pd.DataFrame:
    """
    Compares multiple HPO-selected models on test data for a specific horizon step.

    Args:
        models: List of model names to compare.
        horizon: Forecast horizon (e.g., 4 or 24).
        asset_class: Asset category (crypto, forex, indices).
        shift: Forecast step to evaluate (1-based index).
               Must be between 1 and horizon.
        plot: Whether to generate comparison plots.

    Returns:
        Ranked pd.DataFrame with metrics and ranking.
    """

    # -----------------------------
    # 0. Shift Validation
    # -----------------------------
    if not (1 <= shift <= horizon):
        raise ValueError(
            f"`shift` must be between 1 and {horizon} for horizon={horizon}. "
            f"Got shift={shift}."
        )

    # Convert to 0-based index
    shift_idx = shift - 1

    # -----------------------------
    # 1. Validation
    # -----------------------------
    available_models = _get_available_models()
    valid_models = [m for m in models if m in available_models]
    if not valid_models:
        raise ValueError(
            f"None of the requested models {models} found in HPO results. "
            f"Available: {available_models}"
        )

    if valid_models != models:
        print(f"⚠️ Warning: Following models were not found and will be skipped: "
              f"{set(models) - set(valid_models)}")

    # Resolve representative asset
    try:
        representative = config.get_representative_asset(asset_class)
    except KeyError:
        raise ValueError(
            f"Invalid asset class: {asset_class}. "
            f"Choose from: {config.get_categories()}"
        )

    # -----------------------------
    # 2. Data Loading
    # -----------------------------
    data_dir = config.get_path("data_processed") / asset_class / representative / str(horizon)
    if not data_dir.exists():
        raise FileNotFoundError(f"Processed test data not found at {data_dir}")

    dataset = load_processed_data(data_dir)
    test_x, test_y = dataset["test_x"], dataset["test_y"]

    results = []
    model_predictions = {}

    print(f"📊 Comparing {len(valid_models)} models for {asset_class.upper()} "
          f"({representative}) | Horizon: {horizon} | Shift: t+{shift}")

    # -----------------------------
    # 3. Model Inference & Metrics
    # -----------------------------
    for model_name in valid_models:
        try:
            best_trial = _load_best_trial_num(model_name, asset_class, representative, horizon)
            model, _, device = _load_model_for_trial(
                model_name, asset_class, representative, horizon, best_trial
            )

            trainer = Trainer(model, device, {"batch_size": 128}, seed=42)
            preds = trainer.predict(test_x)

            # Select only the specific horizon step
            y_true_step = test_y[:, shift_idx]
            y_pred_step = preds[:, shift_idx]

            model_predictions[model_name] = y_pred_step

            mse_val = mean_squared_error(y_true_step, y_pred_step)
            rmse_val = np.sqrt(mse_val)
            mae_val = mean_absolute_error(y_true_step, y_pred_step)
            mape_val = mean_absolute_percentage_error(y_true_step, y_pred_step)
            smape_val = _calculate_smape(y_true_step, y_pred_step)
            r2_val = r2_score(y_true_step, y_pred_step)

            results.append({
                "Model": model_name,
                "MAE": mae_val,
                "RMSE": rmse_val,
                "MAPE": mape_val,
                "SMAPE": smape_val,
                "R2": r2_val,
                "errors": (y_true_step - y_pred_step) ** 2
            })

        except Exception as e:
            print(f"   ❌ Failed to evaluate {model_name}: {e}")

    if not results:
        return pd.DataFrame()

    # -----------------------------
    # 4. Ranking & Statistical Test
    # -----------------------------
    df = pd.DataFrame(results)
    df = df.sort_values("RMSE").reset_index(drop=True)
    df["Rank"] = df.index + 1

    best_model_errors = df.iloc[0]["errors"].flatten()
    p_values = [1.0]

    for i in range(1, len(df)):
        comp_errors = df.iloc[i]["errors"].flatten()
        _, p_val = stats.ttest_rel(best_model_errors, comp_errors)
        p_values.append(p_val)

    df["p-value (vs Best)"] = p_values

    display_df = df.drop(columns=["errors"])

    # -----------------------------
    # 5. Visualization
    # -----------------------------
    if plot:
        plt.figure(figsize=(10, 6))
        sns.barplot(x="Model", y="RMSE", data=display_df, palette="viridis")
        plt.title(
            f"RMSE Comparison - Horizon {horizon} (t+{shift})",
            fontweight='bold'
        )
        plt.ylabel("RMSE")
        plt.show()

        # Tracking Plot (Top 2)
        top_n = min(2, len(display_df))
        samples = 200

        plt.figure(figsize=(16, 6))
        y_true_plot = y_true_step[-samples:]
        plt.plot(y_true_plot, label="Actual", color="black", linewidth=2)

        colors = ["crimson", "seagreen"]
        for i in range(top_n):
            m_name = display_df.iloc[i]["Model"]
            y_pred_plot = model_predictions[m_name][-samples:]
            plt.plot(
                y_pred_plot,
                label=f"Rank {i+1}: {m_name}",
                color=colors[i],
                linestyle="--"
            )

        plt.title(
            f"Top {top_n} Models Tracking (Horizon {horizon}, t+{shift})",
            fontweight='bold'
        )
        plt.legend()
        plt.show()

    return display_df


### 🚀 Example Execution
Run the cell below to visualize your HPO results. Feel free to change the parameters to explore different models and trials.

In [ ]:
# Available models: ['Autoformer', 'DLinear', 'LSTM', 'LSTM_Dense', 'ModernTCN', 'N-HiTS', 'PatchTST', 'TimeXer', 'TimesNet', 'iTransformer']
# Categories: ['crypto', 'forex', 'indices']
# Horizons: [4, 24]

In [ ]:
plot_pred(model_name="LSTM_Dense", horizon=24, trial=3, plot_samples=50)

### 🚀 Compare Multiple Models
Run the cell below to perform a comparative analysis between different architectures for a specific asset class and horizon.

In [ ]:
# Available models: ['Autoformer', 'DLinear', 'LSTM', 'LSTM_Dense', 'ModernTCN', 'N-HiTS', 'PatchTST', 'TimeXer', 'TimesNet', 'iTransformer']
# Categories: ['crypto', 'forex', 'indices']
# Horizons: [4, 24]

comparison_df = compare_hpo_models(
    models=[ "PatchTST","TimeXer","DLinear", "ModernTCN", "N-HiTS", "iTransformer"  , 'LSTM_Dense' , 'TimesNet' , 'Autoformer'],
    # models=[ "TimeXer"],
    horizon=24,
    shift=2,
    asset_class="crypto",
)

# Display the ranked results
comparison_df